# DCOPF Example 3 - Economic Dispatch (No Network Constraints)
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (e3 vs e1/e2):** This is a pure economic dispatch formulation. No DC power flow equations, no branch thermal limits, and no voltage angle variables. Only a system-wide power balance constraint is enforced.

In [ ]:
from pyomo.environ import *
import time

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────

# Generators: {index: (bus, Pgmin MW, Pgmax MW, cost $/MWh)}
gen_data = {
    1: {'bus': 1, 'Pgmin': 20, 'Pgmax': 70, 'c': 10},
    2: {'bus': 3, 'Pgmin': 40, 'Pgmax': 90, 'c': 20},
}

# Total system load (MW)
total_load = 100

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.GEN = Set(initialize=gen_data.keys())

# Parameters
model.gen_min    = Param(model.GEN, initialize={g: gen_data[g]['Pgmin'] for g in gen_data})
model.gen_max    = Param(model.GEN, initialize={g: gen_data[g]['Pgmax'] for g in gen_data})
model.gen_OpCost = Param(model.GEN, initialize={g: gen_data[g]['c']     for g in gen_data})

# Decision Variables
model.G = Var(model.GEN)

# ── Objective ────────────────────────────────────────────────────
def obj_rule(model):
    return sum(model.gen_OpCost[g] * model.G[g] for g in model.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Constraints ──────────────────────────────────────────────────
# System-wide power balance
model.SysWidePowerBalance = Constraint(
    expr=sum(model.G[g] for g in model.GEN) == total_load
)

# Generator capacity limits
def genLimits_rule(model, g):
    return inequality(model.gen_min[g], model.G[g], model.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\n=== Generator Dispatch ===')
for g in model.GEN:
    print(f'  G[{g}] (Bus {gen_data[g]["bus"]}) = {model.G[g].value:.4f} MW')

total_cost = sum(gen_data[g]['c'] * model.G[g].value for g in model.GEN)
print(f'\n=== Total Cost = ${total_cost:.2f} ===')
print(f'Total solve elapsed time: {solve_time:.4f} seconds')